In [46]:
import math
import sys
import mpire
import os
os.environ["OMP_NUM_THREADS"] = "1"
import time
import multiprocessing
from mpire import WorkerPool
from pprint import pprint
import itertools
num_cores =max(multiprocessing.cpu_count()//2,1)
import shutil
import random
from multiprocessing import Manager

from ragatouille import RAGPretrainedModel


from sentence_transformers import SentenceTransformer
from voyager import Index, Space

class MyExistingRetrievalPipeline:
    index: Index
    embedder: SentenceTransformer

    def __init__(self, embedder_name: str = "BAAI/bge-small-en-v1.5"):
        self.embedder = SentenceTransformer(embedder_name)
        self.collection_map = {}
        self.index = Index(
            Space.Cosine,
            num_dimensions=self.embedder.get_sentence_embedding_dimension(),
        )

    def index_documents(self, documents: list[str]) -> None:
        # There's very few documents in our example, so we don't bother with batching
        for document in documents:
            self.collection_map[self.index.add_item(self.embedder.encode(document['content']))] = document['content']

    def query(self, query: str, k: int = 10) -> list[str]:
        query_embedding = self.embedder.encode(query)
        to_return = []
        for idx in self.index.query(query_embedding, k=k)[0]:
            to_return.append(self.collection_map[idx])
        return to_return




from ragatouille.utils import get_wikipedia_page
from ragatouille.data import CorpusProcessor


def parallel_process_recipe(get_wikipedia_pages):
    # print(f"get_wikipedia_pages: {get_wikipedia_pages}")
    recipe = get_wikipedia_page(get_wikipedia_pages)
    return {"individual_recipe":recipe}


def create_recipe(recipe_items,top_n_recipes = 2):
    RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
    # RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
    existing_pipeline = MyExistingRetrievalPipeline()

    corpus_processor = CorpusProcessor()
    actual_recipe = "Create a recipe like ratatouille from:-"
    recipe_list = set("Create a recipe like ratatouille from:-".split())

    # recipe_list = [f"in main item {k}, add {v}" for k,v in recipe_items.items()]
    for k,v in recipe_items.items():
      for sub_recipe in v:
        x = f"in {k}, add {sub_recipe}".lower().split()
        actual_recipe+=f"In {k}, add {sub_recipe}.\n"
        # print(x)
        recipe_list.update(x)
        # recipe_list.append(f"in {k}, add {sub_recipe}".lower())
    from pprint import pprint

    # pprint(recipe_list)
    results = [{"get_wikipedia_pages":recipe} for recipe in recipe_list]
    # pprint(results)
    with WorkerPool(n_jobs=num_cores,daemon=False) as pool:
            results = pool.map(parallel_process_recipe, results, progress_bar=True)
    # pprint(results)
    documents = [result["individual_recipe"] for result in results]
    # print(actual_recipe)
    # print(f"docs created")
    # documents = [get_wikipedia_page("Hayao Miyazaki"), get_wikipedia_page("Studio Ghibli"), get_wikipedia_page("Princess Mononoke"), get_wikipedia_page("Shrek")]
    documents = corpus_processor.process_corpus(documents, chunk_size=10000)
    # print(f"docs processed")

    existing_pipeline.index_documents(documents)
    # print(f"docs inserted")


    query = actual_recipe
    # print(f"query = {query}")
    raw_results = existing_pipeline.query(query, k=top_n_recipes)
    return [{f"Tasty Recipe {i}":raw_results} for i in range(1,1+len(raw_results))]

if __name__=="__main__":
    recipe_items = {
        "TOMATO SAUCE BASE":[
            "1 tablespoon olive oil",
            "1 tablespoon unsalted butter",
            "1 tablespoon garlic, minced",
            "1/2 small yellow onion, diced",
            "1 small red bell pepper, diced",
            "4 to 5 sprigs fresh thyme, leaves removed",
            "14 ounces crushed tomatoes",
            "2 tablespoons heavy cream",
            "2 tablespoons parmesan, grated",
            "4 to 5 basil leaves",
            "3/4 teaspoon dried oregano",
            "Salt, to taste",
            "Pepper, to taste",
            "1 teaspoon red vinegar"
        ],
        "VEGETABLES":[
            "1 to 2 zucchinis",
            "4 to 5 small Roma tomatoes",
            "1 to 2 yellow squash",
            "1 to 2 large Chinese eggplants"
        ],
        "ASSEMBLY AND GARNISH":[
            "Olive oil",
            "Salt, to taste",
            "Pepper, to taste",
            "Chopped parsley"
        ]
    }

    final_recipe = create_recipe(recipe_items)
    # pprint(final_recipe)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
100%|██████████| 65/65 [00:10<00:00,  5.94it/s]


In [47]:
pprint(final_recipe)

[{'Tasty Recipe 1': ['Ratatouille ( RAT-ə-TOO-ee, French: [ʁatatuj] ; Occitan: '
                     'ratatolha [ʀataˈtuʎɔ] ) is a traditional French '
                     'vegetable dish originating in the Provence region of '
                     'southern France, particularly associated with Nice and '
                     'its surrounding region. It developed within the context '
                     'of rural Provençal cuisine, where seasonal vegetables '
                     'were stewed together as a practical means of using '
                     'surplus summer produce. The dish consists of a stew or '
                     'sauté of seasonal summer vegetables cooked in olive oil '
                     'and is sometimes referred to as ratatouille niçoise '
                     '(French: [niswaz]).\n'
                     'Although preparation methods and cooking times vary '
                     'considerably by region and household, ratatouille is '
                     'typ

In [18]:
get_wikipedia_page("1")

'1 (one, unit, unity) is a number, numeral, and grapheme. It is the first and smallest positive integer of the infinite sequence of natural numbers. This fundamental property has led to its unique uses in other fields, ranging from science to sports, where it commonly denotes the first, leading, or top thing in a group. 1 is the unit of counting or measurement, and represents a single thing. The representation of 1 evolved from ancient Sumerian and Babylonian symbols to the modern Arabic numeral. Linguistically, in English, "one" is a determiner for singular nouns and a gender-neutral pronoun.\nIn mathematics, 1 is the multiplicative identity, meaning that any number multiplied by 1 equals the same number. 1 is by convention not considered a prime number. In digital technology, 1 represents the "on" state in binary code, the foundation of computing. Philosophically, 1 symbolizes the ultimate reality or source of existence in various traditions.\n\n\n== In mathematics ==\nThe number 1 i

In [4]:
# !pip install -r /content/sample_data/requirements.txt
!pip install "transformers>=4.36,<4.40"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
